In [1]:
# 龙头底分型战法 - Notebook格式（RiceQuant）
# 快速验证核心选股逻辑

print("=" * 80)
print("龙头底分型战法 - Notebook验证")
print("=" * 80)

import pandas as pd
import numpy as np

try:
    # 测试日期范围
    test_dates = ["2024-03-01", "2024-06-01", "2024-09-01", "2024-12-01"]

    all_results = []

    for date in test_dates:
        print(f"\n{'=' * 60}")
        print(f"测试日期: {date}")
        print(f"{'=' * 60}")

        # 1. 获取所有股票（排除科创板、创业板）
        try:
            all_stocks = all_instruments("CS", date)
            stocks = [
                s.order_book_id
                for s in all_stocks
                if not (
                    s.order_book_id.startswith("68")
                    or s.order_book_id.startswith("300")
                )
            ]
            print(f"股票池大小: {len(stocks)}")
        except Exception as e:
            print(f"获取股票池失败: {e}")
            continue

        # 2. 筛选涨停股票
        try:
            limit_up_stocks = []
            sample_size = min(100, len(stocks))  # 限制计算量

            for stock in stocks[:sample_size]:
                try:
                    bars = history_bars(stock, 1, "1d", "close,limit_up", date)
                    if bars is not None and len(bars) > 0:
                        if bars[0]["close"] >= bars[0]["limit_up"] * 0.995:
                            limit_up_stocks.append(stock)
                except:
                    pass

            print(f"涨停股票数: {len(limit_up_stocks)}")

            if len(limit_up_stocks) == 0:
                print("无涨停股票，跳过")
                continue

        except Exception as e:
            print(f"筛选涨停失败: {e}")
            continue

        # 3. 检查底分型形态
        signals = []
        test_count = min(20, len(limit_up_stocks))  # 测试20只

        for stock in limit_up_stocks[:test_count]:
            try:
                # 获取最近3天数据
                bars_3 = history_bars(
                    stock, 3, "1d", "open,close,high,low,limit_up", date
                )
                if bars_3 is None or len(bars_3) < 3:
                    continue

                # 获取60日数据
                bars_60 = history_bars(stock, 60, "1d", "close", date)
                if bars_60 is None or len(bars_60) < 60:
                    continue

                ma60 = np.mean(bars_60["close"])

                # T-2日（十字星）
                t2_close = bars_3[1]["close"]
                t2_open = bars_3[1]["open"]
                t2_high = bars_3[1]["high"]
                t2_low = bars_3[1]["low"]

                body_ratio = abs(t2_close - t2_open) / ((t2_close + t2_open) / 2)
                swing_ratio = abs(t2_high - t2_low) / ((t2_high + t2_low) / 2)

                # T-1日（涨停）
                t1_close = bars_3[2]["close"]
                t1_open = bars_3[2]["open"]
                t1_limit_up = bars_3[2]["limit_up"]

                # 底分型条件
                is_doji = body_ratio < 0.03 and swing_ratio < 0.10
                above_ma60 = t2_close > ma60
                is_limit_up = t1_close >= t1_limit_up * 0.995
                gap_up = t1_open > t2_close * 1.02
                strong_close = t1_close > t2_close * 1.05

                if is_doji and above_ma60 and is_limit_up and gap_up and strong_close:
                    signals.append(
                        {
                            "stock": stock,
                            "date": date,
                            "t2_close": t2_close,
                            "t1_close": t1_close,
                            "return_pct": (t1_close / t2_close - 1) * 100,
                        }
                    )
                    print(
                        f"发现信号: {stock} T-2收{t2_close:.2f} T-1收{t1_close:.2f} 涨幅{return_pct:.2f}%"
                    )

            except Exception as e:
                continue

        print(f"\n信号数量: {len(signals)}")

        if len(signals) > 0:
            avg_return = np.mean([s["return_pct"] for s in signals])
            print(f"平均涨幅: {avg_return:.2f}%")
            all_results.extend(signals)

    # 4. 总体统计
    print(f"\n{'=' * 80}")
    print("总体统计")
    print(f"{'=' * 80}")
    print(f"测试日期数: {len(test_dates)}")
    print(f"总信号数: {len(all_results)}")

    if len(all_results) > 0:
        avg_return = np.mean([r["return_pct"] for r in all_results])
        max_return = np.max([r["return_pct"] for r in all_results])
        min_return = np.min([r["return_pct"] for r in all_results])

        print(f"平均涨幅: {avg_return:.2f}%")
        print(f"最大涨幅: {max_return:.2f}%")
        print(f"最小涨幅: {min_return:.2f}%")

        # 显示信号列表
        print(f"\n信号列表:")
        for r in all_results:
            print(f"  {r['stock']} @ {r['date']}: {r['return_pct']:.2f}%")
    else:
        print("无信号发现")

except Exception as e:
    print(f"执行错误: {e}")
    import traceback

    traceback.print_exc()

print("\n" + "=" * 80)
print("验证完成")
print("=" * 80)


龙头底分型战法 - Notebook验证

测试日期: 2024-03-01


获取股票池失败: 'str' object has no attribute 'order_book_id'

测试日期: 2024-06-01


获取股票池失败: 'str' object has no attribute 'order_book_id'

测试日期: 2024-09-01


获取股票池失败: 'str' object has no attribute 'order_book_id'

测试日期: 2024-12-01


获取股票池失败: 'str' object has no attribute 'order_book_id'

总体统计
测试日期数: 4
总信号数: 0
无信号发现

验证完成
